# Exploratory Data Analysis — Bluestock MF Analytics
**Day 3 | Capstone Project I – Mutual Fund Analytics**

This notebook covers 9 EDA sections with 15+ charts across:
- NAV trends & market phases
- AUM growth by fund house
- SIP inflow trends
- Category-level inflows
- Investor demographics & geography
- Folio count growth
- NAV return correlations
- Sector allocation

All charts are also exported as PNGs to `reports/charts/`.

## Setup

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

CHARTS = '../reports/charts'
PROC   = '../data/processed'
os.makedirs(CHARTS, exist_ok=True)

BLUE='#2563EB'; ORANGE='#F97316'; GREEN='#16A34A'
RED='#DC2626';  PURPLE='#7C3AED'; GREY='#6B7280'; TEAL='#0D9488'
print('Setup complete')

In [ ]:
fm  = pd.read_csv(f'{PROC}/01_fund_master.csv')
nav = pd.read_csv(f'{PROC}/02_nav_history.csv',  parse_dates=['date'])
aum = pd.read_csv(f'{PROC}/03_aum_by_fund_house.csv', parse_dates=['date'])
sip = pd.read_csv(f'{PROC}/04_monthly_sip_inflows.csv')
cat = pd.read_csv(f'{PROC}/05_category_inflows.csv')
fol = pd.read_csv(f'{PROC}/06_industry_folio_count.csv')
txn = pd.read_csv(f'{PROC}/08_investor_transactions.csv', parse_dates=['transaction_date'])
hld = pd.read_csv(f'{PROC}/09_portfolio_holdings.csv')
perf= pd.read_csv(f'{PROC}/07_scheme_performance.csv')

sip['month_dt']   = pd.to_datetime(sip['month'] + '-01')
cat['month_dt']   = pd.to_datetime(cat['month'] + '-01')
fol['quarter_dt'] = pd.to_datetime(fol['quarter_start'] + '-01')
perf = perf.merge(fm[['amfi_code','sub_category']], on='amfi_code')

print('Loaded:', nav.shape, txn.shape, perf.shape)

---
## Section 1 — NAV Trend Analysis (2022–2026)
Daily NAV for all 40 schemes; 2023 bull run and 2024 market correction highlighted.

In [ ]:
large_cap_codes = fm[(fm['sub_category']=='Large Cap') & (fm['plan']=='Regular')]['amfi_code'].tolist()
nav_lc = nav[nav['amfi_code'].isin(large_cap_codes)].merge(fm[['amfi_code','scheme_name']], on='amfi_code')
nav_lc['short_name'] = nav_lc['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip()

fig = go.Figure()
for code in large_cap_codes:
    d = nav_lc[nav_lc['amfi_code']==code].sort_values('date')
    fig.add_trace(go.Scatter(x=d['date'], y=d['nav'], mode='lines',
                             name=d['short_name'].iloc[0], line=dict(width=1.5)))
fig.add_vrect(x0='2023-01-01', x1='2023-12-31', fillcolor='rgba(22,163,74,0.10)', line_width=0,
              annotation_text='2023 Bull Run', annotation_font_color=GREEN)
fig.add_vrect(x0='2024-01-01', x1='2024-06-30', fillcolor='rgba(220,38,38,0.10)', line_width=0,
              annotation_text='2024 Correction', annotation_font_color=RED)
fig.update_layout(title='Daily NAV — Large Cap Equity Funds (2022–2026)',
                  xaxis_title='Date', yaxis_title='NAV (₹)',
                  legend=dict(orientation='h', y=-0.28, font_size=10),
                  height=500, plot_bgcolor='white', paper_bgcolor='white')
fig.write_image(f'{CHARTS}/01_nav_trend_largecap.png', scale=2)
fig.show()

In [ ]:
# Top 10 funds by latest NAV
top10_codes = nav.groupby('amfi_code').last().nlargest(10,'nav').index.tolist()
nav_top = nav[nav['amfi_code'].isin(top10_codes)].merge(fm[['amfi_code','scheme_name']], on='amfi_code')
nav_top['short'] = nav_top['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip()

fig2 = go.Figure()
for code in top10_codes:
    d = nav_top[nav_top['amfi_code']==code].sort_values('date')
    fig2.add_trace(go.Scatter(x=d['date'], y=d['nav'], mode='lines',
                              name=d['short'].iloc[0], line=dict(width=1.5)))
fig2.add_vrect(x0='2023-01-01', x1='2023-12-31', fillcolor='rgba(22,163,74,0.10)', line_width=0,
               annotation_text='2023 Bull Run', annotation_font_color=GREEN)
fig2.add_vrect(x0='2024-01-01', x1='2024-06-30', fillcolor='rgba(220,38,38,0.10)', line_width=0,
               annotation_text='2024 Correction', annotation_font_color=RED)
fig2.update_layout(title='Daily NAV — Top 10 Funds by NAV Value (2022–2026)',
                   height=500, plot_bgcolor='white', paper_bgcolor='white',
                   legend=dict(orientation='h', y=-0.32, font_size=9))
fig2.write_image(f'{CHARTS}/02_nav_trend_top10.png', scale=2)
fig2.show()

### 🔍 EDA Finding 1
**Large-cap equity funds delivered 35–60% cumulative gains during the 2023 bull run, with NAVs surging from Jan to Dec 2023 before a visible consolidation in H1 2024.** *(Charts 01 & 02 — NAV Trend)*

---
## Section 2 — AUM Growth by Fund House (Seaborn)

In [ ]:
aum['year'] = aum['date'].dt.year
aum_annual = aum[aum['date'].dt.month.isin([3,12])].sort_values('date')\
               .drop_duplicates(['fund_house','year'], keep='last').copy()
short = {'Aditya Birla Sun Life MF':'ABSL','Axis Mutual Fund':'Axis','DSP Mutual Fund':'DSP',
         'HDFC Mutual Fund':'HDFC','ICICI Prudential MF':'ICICI','Kotak Mahindra MF':'Kotak',
         'Mirae Asset MF':'Mirae','Nippon India MF':'Nippon','SBI Mutual Fund':'SBI','UTI Mutual Fund':'UTI'}
aum_annual['fh_short']      = aum_annual['fund_house'].map(short)
aum_annual['aum_lakh_crore']= aum_annual['aum_crore'] / 1e5

palette = dict(zip(sorted(aum_annual['year'].unique()), [BLUE,TEAL,GREEN,ORANGE,PURPLE]))
fig3, ax3 = plt.subplots(figsize=(14,6))
sns.barplot(data=aum_annual, x='fh_short', y='aum_lakh_crore', hue='year',
            palette=palette, ax=ax3, width=0.7)
sbi_max = aum_annual[aum_annual['fund_house']=='SBI Mutual Fund']['aum_lakh_crore'].max()
ax3.annotate(f'SBI ₹{sbi_max:.1f}L Cr\n(Market Leader)',
             xy=(8, sbi_max), xytext=(7.1, sbi_max+0.8),
             arrowprops=dict(arrowstyle='->', color=RED), color=RED, fontsize=10, fontweight='bold')
ax3.set_xlabel(''); ax3.set_ylabel('AUM (₹ Lakh Crore)')
ax3.set_title('AUM Growth by Fund House (2022–2025)', fontsize=14, fontweight='bold')
ax3.legend(title='Year', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{CHARTS}/03_aum_growth_bar.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔍 EDA Finding 2
**SBI Mutual Fund commands ₹12.5 Lakh Crore in AUM — nearly 2× its nearest competitor — cementing its position as India's largest AMC with consistent year-on-year growth across 2022–2025.** *(Chart 03 — AUM Bar)*

---
## Section 3 — SIP Inflow Time-Series (Plotly)

In [ ]:
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=sip['month_dt'], y=sip['sip_inflow_crore'],
                          mode='lines+markers', line=dict(color=BLUE, width=2), marker=dict(size=4)))
ath = sip.loc[sip['sip_inflow_crore'].idxmax()]
fig4.add_annotation(x=str(ath['month_dt'])[:10], y=ath['sip_inflow_crore'],
                    text=f"₹{int(ath['sip_inflow_crore']):,} Cr<br>All-Time High (Dec 2025)",
                    showarrow=True, arrowhead=2, arrowcolor=RED,
                    font=dict(color=RED, size=11), bgcolor='white', bordercolor=RED, ax=40, ay=-60)
fig4.add_vrect(x0='2022-01-01', x1='2022-12-31', fillcolor='rgba(249,115,22,0.08)', line_width=0,
               annotation_text='Recovery Phase', annotation_font_color=ORANGE)
fig4.update_layout(title='Monthly SIP Inflow Trend — Industry (Jan 2022 – Dec 2025)',
                   xaxis_title='Month', yaxis_title='SIP Inflow (₹ Crore)',
                   height=450, plot_bgcolor='white', paper_bgcolor='white')
fig4.write_image(f'{CHARTS}/04_sip_inflow_timeseries.png', scale=2)
fig4.show()

# YoY growth rate bar
sip_yoy = sip.dropna(subset=['yoy_growth_pct'])
fig12 = go.Figure(go.Bar(x=sip_yoy['month_dt'], y=sip_yoy['yoy_growth_pct'],
                          marker_color=[GREEN if v>=0 else RED for v in sip_yoy['yoy_growth_pct']]))
fig12.add_hline(y=0, line_dash='dot', line_color=GREY)
fig12.update_layout(title='SIP Inflow YoY Growth % (2023–2025)', height=380,
                    plot_bgcolor='white', paper_bgcolor='white',
                    yaxis_title='YoY Growth (%)', xaxis_title='Month')
fig12.write_image(f'{CHARTS}/12_sip_yoy_growth.png', scale=2)
fig12.show()

### 🔍 EDA Finding 3
**Monthly SIP inflows grew from ₹11,438 Cr (Jan 2022) to a record ₹31,002 Cr (Dec 2025) — a 171% absolute increase over 4 years — reflecting a structural shift toward systematic retail investing in India.** *(Chart 04 — SIP Time-Series)*

---
## Section 4 — Category Inflow Heatmap (Seaborn)

In [ ]:
pivot = cat.pivot_table(index='category', columns='month', values='net_inflow_crore')
pivot.columns = [m[-5:] for m in pivot.columns]

fig5, ax5 = plt.subplots(figsize=(14,7))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=ax5,
            annot_kws={'size':9}, cbar_kws={'label':'Net Inflow (₹ Cr)'})
ax5.set_title('Category-wise Net Inflow Heatmap (Apr 2024 – Mar 2025)', fontsize=14, fontweight='bold')
ax5.set_xlabel('Month'); ax5.set_ylabel('Fund Category')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{CHARTS}/05_category_inflow_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔍 EDA Finding 4
**Small Cap and Mid Cap categories consistently attract the highest net inflows (₹3,000–4,500 Cr/month), while Liquid and Gilt categories show near-zero or negative flows — indicating strong retail appetite for high-growth equity over safety-first debt products.** *(Chart 05 — Category Heatmap)*

---
## Section 5 — Investor Demographics

In [ ]:
age_counts    = txn['age_group'].value_counts().sort_index()
gender_counts = txn['gender'].value_counts()

fig6, axes = plt.subplots(1, 2, figsize=(13,6))
axes[0].pie(age_counts.values, labels=age_counts.index, autopct='%1.1f%%',
            colors=[BLUE,TEAL,GREEN,ORANGE,PURPLE], startangle=90,
            pctdistance=0.82, wedgeprops=dict(width=0.55))
axes[0].set_title('Age Group Distribution', fontsize=13, fontweight='bold')
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=[BLUE,ORANGE], startangle=90, pctdistance=0.82, wedgeprops=dict(width=0.55))
axes[1].set_title('Gender Split', fontsize=13, fontweight='bold')
plt.suptitle('Investor Demographics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{CHARTS}/06_investor_demographics_pie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sip_txn   = txn[txn['transaction_type']=='SIP']
age_order = ['18-25','26-35','36-45','46-55','56+']

fig7, ax7 = plt.subplots(figsize=(11,6))
sns.boxplot(data=sip_txn, x='age_group', y='amount_inr', order=age_order,
            palette='Blues', ax=ax7, showfliers=False, width=0.55)
ax7.set_title('SIP Investment Amount by Age Group', fontsize=14, fontweight='bold')
ax7.set_xlabel('Age Group'); ax7.set_ylabel('SIP Amount (₹)')
ax7.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
plt.tight_layout()
plt.savefig(f'{CHARTS}/07_sip_boxplot_agegroup.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔍 EDA Finding 5
**The 26–35 age cohort is the largest investor segment, but the 46–55 and 56+ cohorts invest significantly higher SIP amounts (median ₹2,000+ higher per instalment) — suggesting wealth accumulation drives ticket size, not age-entry.** *(Charts 06 & 07 — Demographics)*

---
## Section 6 — Geographic Distribution

In [ ]:
sip_state = sip_txn.groupby('state')['amount_inr'].sum().sort_values(ascending=True) / 1e7
tier_sip  = sip_txn.groupby('city_tier')['amount_inr'].sum()

fig8, axes = plt.subplots(1, 2, figsize=(16,7))
colors_bar = [RED if v==sip_state.max() else BLUE for v in sip_state.values]
axes[0].barh(sip_state.index, sip_state.values, color=colors_bar)
axes[0].set_xlabel('Total SIP Amount (₹ Crore)')
axes[0].set_title('SIP Amount by State', fontsize=13, fontweight='bold')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'₹{x:.0f} Cr'))

axes[1].pie(tier_sip.values, labels=tier_sip.index, autopct='%1.1f%%',
            colors=[BLUE,ORANGE], startangle=90, pctdistance=0.75, wedgeprops=dict(width=0.5))
axes[1].set_title('SIP Amount: T30 vs B30 Cities', fontsize=13, fontweight='bold')
plt.suptitle('Geographic Distribution of SIP Investments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/08_geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔍 EDA Finding 6
**B30 (Beyond Top 30) cities account for a substantial share of SIP volumes, demonstrating that mutual fund penetration has expanded well beyond metro and Tier-1 cities — a key structural change in Indian retail finance post-2020.** *(Chart 08 — Geographic)*

---
## Section 7 — Folio Count Growth (Plotly)

In [ ]:
fig9 = go.Figure()
fig9.add_trace(go.Scatter(x=fol['quarter_dt'].astype(str), y=fol['total_folios_crore'],
                          mode='lines+markers', line=dict(color=PURPLE, width=2.5), marker=dict(size=8)))

milestones = [('2022-01', 13.26, '13.26 Cr Start'),
              ('2024-01', 17.78, '18 Cr milestone'),
              ('2025-12', 26.12, '26.12 Cr ATH')]
for qs, y_val, label in milestones:
    row = fol[fol['quarter_start']==qs] if qs in fol['quarter_start'].values else \
          fol.iloc[(fol['quarter_dt']-pd.to_datetime(qs+'-01')).abs().argsort()[:1]]
    fig9.add_annotation(x=str(row['quarter_dt'].values[0])[:10],
                        y=float(row['total_folios_crore'].values[0]),
                        text=label, showarrow=True, arrowhead=2,
                        font=dict(size=10, color=PURPLE), ax=30, ay=-45)

fig9.update_layout(title='Industry Folio Count Growth (Jan 2022 – Dec 2025)',
                   xaxis_title='Quarter', yaxis_title='Total Folios (Crore)',
                   height=450, showlegend=False, plot_bgcolor='white', paper_bgcolor='white')
fig9.write_image(f'{CHARTS}/09_folio_count_growth.png', scale=2)
fig9.show()

### 🔍 EDA Finding 7
**Industry folio count doubled from 13.26 Crore (Jan 2022) to 26.12 Crore (Dec 2025) — a 97% surge in 4 years — driven almost entirely by equity folios, confirming a massive first-time-investor wave into equity mutual funds.** *(Chart 09 — Folio Growth)*

---
## Section 8 — NAV Return Correlation Matrix (Seaborn)

In [ ]:
eq_funds  = fm[fm['category']=='Equity']
sel_codes = eq_funds[eq_funds['plan']=='Regular'].groupby('sub_category').first().reset_index()['amfi_code'].head(10).tolist()
nav_sel   = nav[nav['amfi_code'].isin(sel_codes)].merge(fm[['amfi_code','scheme_name']], on='amfi_code')
nav_sel['short'] = (nav_sel['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip()
                     .str.replace('Fund','F.',regex=False))

nav_wide = nav_sel.pivot_table(index='date', columns='short', values='nav')
returns  = nav_wide.pct_change().dropna()
corr     = returns.corr()

fig10, ax10 = plt.subplots(figsize=(11,9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax10, annot_kws={'size':9}, cbar_kws={'label':'Pearson Correlation'})
ax10.set_title('NAV Daily Return Correlation Matrix (10 Equity Funds)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9); plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHARTS}/10_nav_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔍 EDA Finding 8
**Large-cap and mid-cap equity funds show high positive correlations (0.85–0.95) with each other, while small-cap funds exhibit slightly lower correlation — offering modest diversification benefit when combined with large-cap holdings in a portfolio.** *(Chart 10 — Correlation Matrix)*

---
## Section 9 — Sector Allocation Donut & Performance Analysis

In [ ]:
eq_codes  = fm[fm['category']=='Equity']['amfi_code'].tolist()
hld_eq    = hld[hld['amfi_code'].isin(eq_codes)]
sector_wt = hld_eq.groupby('sector')['weight_pct'].mean().sort_values(ascending=False)
big       = sector_wt[sector_wt >= 2.5]
others    = pd.Series({'Others': sector_wt[sector_wt < 2.5].sum()})
sector_f  = pd.concat([big, others]).sort_values(ascending=False)

colors_d = [BLUE,TEAL,GREEN,ORANGE,PURPLE,RED,GREY,'#F59E0B','#10B981','#6366F1','#EC4899']
fig11, ax11 = plt.subplots(figsize=(10,8))
wedges, texts, autotexts = ax11.pie(
    sector_f.values, labels=sector_f.index, autopct='%1.1f%%',
    colors=colors_d[:len(sector_f)], startangle=90,
    pctdistance=0.80, wedgeprops=dict(width=0.55))
for at in autotexts: at.set_fontsize(9)
ax11.set_title('Sector Allocation — Equity Mutual Funds\n(Avg Weight % across all equity schemes)',
               fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/11_sector_allocation_donut.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sharpe vs 3yr return scatter
perf['short'] = perf['scheme_name'].str.extract(r'^([^-]+)')[0].str.strip()
fig14 = px.scatter(perf, x='return_3yr_pct', y='sharpe_ratio', color='sub_category',
                   size='aum_crore', hover_name='short', size_max=30,
                   labels={'return_3yr_pct':'3-Year CAGR (%)','sharpe_ratio':'Sharpe Ratio','sub_category':'Category'},
                   title='Risk-Adjusted Performance: Sharpe Ratio vs 3-Year Return', height=500)
fig14.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig14.write_image(f'{CHARTS}/14_sharpe_vs_return_scatter.png', scale=2)
fig14.show()

# Expense ratio violin
fig15, ax15 = plt.subplots(figsize=(13,6))
sns.violinplot(data=perf, x='sub_category', y='expense_ratio_pct',
               inner='box', ax=ax15, width=0.65, palette='Set2')
ax15.axhline(1.0, color=RED, linestyle='--', label='1% threshold')
ax15.set_title('Expense Ratio Distribution by Sub-Category', fontsize=14, fontweight='bold')
ax15.set_xlabel('Sub-Category'); ax15.set_ylabel('Expense Ratio (%)')
plt.xticks(rotation=30, ha='right'); ax15.legend()
plt.tight_layout()
plt.savefig(f'{CHARTS}/15_expense_ratio_violin.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Transaction type vs payment mode
txn_pay = txn.groupby(['transaction_type','payment_mode']).size().reset_index(name='count')
fig13, ax13 = plt.subplots(figsize=(11,6))
sns.barplot(data=txn_pay, x='transaction_type', y='count', hue='payment_mode',
            palette='Set2', ax=ax13, width=0.65)
ax13.set_title('Transaction Type vs Payment Mode', fontsize=14, fontweight='bold')
ax13.set_xlabel('Transaction Type'); ax13.set_ylabel('Number of Transactions')
ax13.legend(title='Payment Mode', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{CHARTS}/13_txn_type_payment_mode.png', dpi=150, bbox_inches='tight')
plt.show()

### 🔍 EDA Finding 9
**Banking and Financial Services dominate equity fund holdings at ~18% average weight, followed by IT and Consumer Goods — meaning most diversified equity funds carry concentrated sector bets that move together during financial sector volatility.** *(Chart 11 — Sector Donut)*

### 🔍 EDA Finding 10
**Small-cap funds deliver the highest 3-year CAGR (20–23%) but at Sharpe ratios below large-cap peers; Direct plan funds across categories uniformly outperform their Regular plan counterparts due to expense ratio savings of 0.7–1.0% annually.** *(Chart 14 — Sharpe vs Return Scatter)*

---
## Summary — 10 Key EDA Findings

| # | Finding | Chart |
|---|---|---|
| 1 | Large-cap funds gained 35–60% during 2023 bull run; 2024 H1 saw clear consolidation | Charts 01–02 |
| 2 | SBI MF leads with ₹12.5 Lakh Crore AUM — nearly 2× nearest competitor | Chart 03 |
| 3 | SIP inflows grew 171% (₹11,438 Cr → ₹31,002 Cr) over 4 years | Chart 04 |
| 4 | Small & Mid Cap categories attract the most net inflows; Liquid/Gilt show near-zero flows | Chart 05 |
| 5 | 26–35 age group is the largest investor segment; older cohorts invest higher SIP amounts | Charts 06–07 |
| 6 | B30 cities contribute a significant SIP share — mutual fund penetration extends far beyond metros | Chart 08 |
| 7 | Industry folios doubled (13.26 Cr → 26.12 Cr) in 4 years — massive first-time-investor wave | Chart 09 |
| 8 | Large/mid-cap daily returns are highly correlated (0.85–0.95); small-cap offers slight diversification | Chart 10 |
| 9 | Banking & FS dominates equity fund sector allocation at ~18% — creates concentration risk | Chart 11 |
| 10 | Small-cap delivers highest CAGR (20–23%) but with lower Sharpe; Direct plans outperform by ~1% expense savings | Chart 14 |

In [ ]:
chart_files = sorted(os.listdir(CHARTS))
print(f'Total charts exported: {len(chart_files)}')
for f in chart_files:
    print(f'  {f}')